# Logistic regression: baseline, tuning, coefficients, OOF

In [1]:
import sys, json
sys.path.insert(0, "..")
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from src.config import ROOT, SEED
from src.data import load_train
from src.evaluate import cv_evaluate
from src.models import make_pipeline
from src.preprocessing import EXCLUDED_COLS

In [2]:
df = load_train()
X, y = df.drop(columns=EXCLUDED_COLS), df["label"]
pipe = make_pipeline("lr", X)
cv_evaluate("lr_baseline", pipe, X, y)

{'model': 'lr_baseline',
 'split': 'cv',
 'accuracy': 0.9330390624296887,
 'accuracy_std': 0.001137206596592781,
 'precision': 0.9184213087812279,
 'precision_std': 0.0018123573815812637,
 'recall': 0.9895174289516693,
 'recall_std': 0.0009112100974001216,
 'f1': 0.9526430057184848,
 'f1_std': 0.0007500703833946626,
 'fpr': 0.18732142857142858,
 'fpr_std': 0.0046599958395356755,
 'roc_auc': 0.982020059318215,
 'roc_auc_std': 0.0010075687676951645,
 'pr_auc': 0.9905890294313238,
 'pr_auc_std': 0.0006095016982610965,
 'fit_seconds': 52.76020383834839,
 'fit_seconds_std': 1.6841965037960411}

In [3]:
grid = {"model__C": [0.01, 0.1, 1, 10, 100],
        "model__penalty": ["l1", "l2"],
        "model__class_weight": [None, "balanced"]}
search = GridSearchCV(pipe, grid, scoring="f1", n_jobs=-1,
                      cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
search.fit(X, y)
(ROOT / "configs" / "lr.json").write_text(json.dumps(search.best_params_, indent=2))
search.best_params_, search.best_score_

({'model__C': 10, 'model__class_weight': None, 'model__penalty': 'l2'},
 0.9527467562355383)

In [4]:
cv_evaluate("lr_tuned", make_pipeline("lr", X).set_params(**search.best_params_), X, y)

{'model': 'lr_tuned',
 'split': 'cv',
 'accuracy': 0.9331930483196121,
 'accuracy_std': 0.0011576046533422401,
 'precision': 0.9186141469373874,
 'precision_std': 0.001809447465489338,
 'recall': 0.9895174286006106,
 'recall_std': 0.0009223192028827435,
 'f1': 0.9527467562355383,
 'f1_std': 0.0007661926881806083,
 'fpr': 0.1868392857142857,
 'fpr_std': 0.004647868174919399,
 'roc_auc': 0.9820612065089342,
 'roc_auc_std': 0.0010148429329265264,
 'pr_auc': 0.9906088228958019,
 'pr_auc_std': 0.0006124032544483619,
 'fit_seconds': 46.66549167633057,
 'fit_seconds_std': 1.8394954948175681}

In [5]:
best = search.best_estimator_
coef = pd.DataFrame({"feature": best.named_steps["prep"].get_feature_names_out(),
                     "coef": best.named_steps["model"].coef_[0]})
top = coef.reindex(coef["coef"].abs().sort_values(ascending=False).index).head(15)
top.to_csv(ROOT / "results" / "lr_coefficients.csv", index=False)
top

,feature,coef
7,num__dttl,4.501840
188,cat__state_INT,3.785553
16,num__swin,-3.570732
158,cat__proto_udp,-3.000166
181,cat__service_smtp,2.436964
9,num__dload,-2.246051
174,cat__service_dns,1.994021
172,cat__service_-,-1.812487
24,num__dmean,1.694559
185,cat__state_CON,-1.582717


In [6]:
from src.oof import save_oof
save_oof("lr")

WindowsPath('E:/uni/year 4/Sem 1/ML/Project/results/oof_lr.csv')